# Train drilling advisory near_5 model

Notebook обучает только `near_5`.

Упрощения:

- нет fallback-путей и больших `try/except`;
- входные `d_depth` и `d_time` берутся из `telemetry_with_energy_quantiles.csv`;
- `depth_m` не используется;
- well-level non-overlap replay строит candidate grid векторно по всем тестовым блокам.


Well-level time report использует **non-overlap near_5**: для оценки времени берутся непересекающиеся блоки по 5 следующих шагов.


In [ ]:
from pathlib import Path
import json
import sys
import warnings
warnings.filterwarnings("ignore")

import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from config import (
    RANDOM_STATE,
    EPS,
    GRID_SIZE,
    MAX_DELTA_FRAC,
    FINAL_OPTIMIZER_MODE,
    CHANGE_PENALTY_WEIGHT,
    BOUNDARY_PENALTY_WEIGHT,
    BOUNDARY_START,
)

DATA_PATH = PROJECT_ROOT / "notebooks" / "telemetry_with_energy_quantiles.csv"
ARTIFACT_DIR = PROJECT_ROOT / "notebooks" / "model_artifacts"
REPORT_DIR = PROJECT_ROOT / "notebooks" / "model_reports"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

TARGET_HORIZONS = [5]
DEFAULT_HORIZON = 5

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PATH:", DATA_PATH)
print("ARTIFACT_DIR:", ARTIFACT_DIR)
print("REPORT_DIR:", REPORT_DIR)
print("GRID_SIZE:", GRID_SIZE)


In [ ]:
DATA_PATH = Path(DATA_PATH)
if not DATA_PATH.exists():
    fallback_data_path = PROJECT_ROOT / "notebooks" / "telemetry_with_energy_quantiles.csv"
    if fallback_data_path.exists():
        DATA_PATH = fallback_data_path
    else:
        raise FileNotFoundError(f"DATA_PATH does not exist: {DATA_PATH}")

df = pd.read_csv(DATA_PATH).drop(columns=["Unnamed: 0"], errors="ignore")

df["processing_time"] = pd.to_datetime(df["processing_time"])
df = df.sort_values(["well_id", "processing_time"]).reset_index(drop=True)
df["rock_energy_type_final"] = df["rock_energy_type_final"].fillna("unknown").astype(str)

numeric_cols = [
    "depth",
    "d_depth",
    "d_time",
    "pressure_axis",
    "pressure_rotation",
    "rotation",
    "speed",
    "hardness_score_smooth",
]
df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric)

print("DATA_PATH:", DATA_PATH)
print("Loaded:", df.shape)
print("Wells:", df["well_id"].nunique())
display(df.head())
display(
    df[
        [
            "d_depth",
            "d_time",
            "pressure_axis",
            "pressure_rotation",
            "rotation",
            "speed",
            "hardness_score_smooth",
        ]
    ].describe(percentiles=[.01, .05, .5, .95, .99])
)


In [ ]:
def add_features(data: pd.DataFrame) -> pd.DataFrame:
    out = data.copy()

    out["dt"] = out["d_time"]

    out["total_pressure"] = out["pressure_axis"] + out["pressure_rotation"]
    out["pressure_balance"] = out["pressure_axis"] / (out["total_pressure"] + EPS)
    out["rotation_efficiency"] = out["rotation"] / (out["pressure_rotation"] + EPS)
    out["axis_x_rotation"] = out["pressure_axis"] * out["rotation"]
    out["energy_input_proxy"] = out["pressure_axis"] + out["pressure_rotation"] * out["rotation"]

    history_cols = [
        "pressure_axis",
        "pressure_rotation",
        "rotation",
        "speed",
    ]

    for col in history_cols:
        grp = out.groupby("well_id")[col]

        for lag in [1, 3, 6]:
            out[f"{col}_lag{lag}"] = grp.shift(lag)

        shifted = grp.shift(1)
        for w in [12]:
            out[f"{col}_roll_mean_{w}"] = (
                shifted.groupby(out["well_id"])
                .rolling(w, min_periods=4)
                .mean()
                .reset_index(level=0, drop=True)
            )
            out[f"{col}_roll_std_{w}"] = (
                shifted.groupby(out["well_id"])
                .rolling(w, min_periods=4)
                .std()
                .reset_index(level=0, drop=True)
            )

    for col in ["pressure_axis", "pressure_rotation", "rotation", "speed"]:
        out[f"{col}_diff1"] = out[col] - out.groupby("well_id")[col].shift(1)

    return out


df = add_features(df)
display(df.head())


In [ ]:
def future_mean_by_group(data: pd.DataFrame, value_col: str, horizon: int) -> pd.Series:
    return (
        data.groupby("well_id")[value_col]
        .transform(
            lambda s: (
                s.shift(-1)
                .rolling(horizon, min_periods=horizon)
                .mean()
                .shift(-(horizon - 1))
            )
        )
    )


target_cols_by_horizon = {}

for h in TARGET_HORIZONS:
    suffix = f"near{h}"

    target_cols_by_horizon[h] = {
        "target_rotation": f"target_rotation_{suffix}",
        "target_speed": f"target_speed_{suffix}",
    }

    for value_col, target_col in [
        ("rotation", target_cols_by_horizon[h]["target_rotation"]),
        ("speed", target_cols_by_horizon[h]["target_speed"]),
    ]:
        df[target_col] = future_mean_by_group(df, value_col, h)

display(
    df[
        [
            "rotation",
            "speed",
            "target_rotation_near5",
            "target_speed_near5",
        ]
    ].describe(percentiles=[.01, .05, .5, .95, .99])
)


In [ ]:
HARDNESS_FEATURE_COLUMNS = ["hardness_score_smooth"]

base_numeric_features = [
    "pressure_axis",
    "pressure_rotation",
    "pressure_balance",
    "rotation",
    "speed",
    "dt",
    "rotation_efficiency",
    "axis_x_rotation",
    "energy_input_proxy",

    *HARDNESS_FEATURE_COLUMNS,

    "pressure_axis_lag1",
    "pressure_axis_lag3",
    "pressure_axis_lag6",

    "rotation_lag1",
    "rotation_lag3",
    "rotation_lag6",

    "speed_lag1",
    "speed_lag3",

    "pressure_axis_roll_mean_12",
    "pressure_axis_roll_std_12",
    "pressure_rotation_roll_mean_12",
    "pressure_rotation_roll_std_12",
    "rotation_roll_mean_12",
    "rotation_roll_std_12",
    "speed_roll_mean_12",
    "speed_roll_std_12",

    "pressure_axis_diff1",
    "pressure_rotation_diff1",
    "rotation_diff1",
    "speed_diff1",
]

categorical_features = ["rock_energy_type_final"]
speed_extra_features = ["candidate_target_rotation"]
speed_numeric_features = base_numeric_features + speed_extra_features


all_target_cols = []
for cols in target_cols_by_horizon.values():
    all_target_cols.extend(cols.values())

all_required = base_numeric_features + categorical_features + all_target_cols
model_df = df.dropna(subset=all_required + ["well_id", "processing_time", "d_depth", "d_time"]).copy()

print("Model df:", model_df.shape)
print("Numeric features:", len(base_numeric_features))
print("Categorical features:", categorical_features)
print("Hardness features:", HARDNESS_FEATURE_COLUMNS)


In [ ]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(model_df, groups=model_df["well_id"]))

train_df = model_df.iloc[train_idx].copy()
test_df = model_df.iloc[test_idx].copy()

print("Train:", train_df.shape, "wells:", train_df["well_id"].nunique())
print("Test:", test_df.shape, "wells:", test_df["well_id"].nunique())


In [ ]:
def make_regressor(seed_offset: int = 0):
    return lgb.LGBMRegressor(
        n_estimators=650,
        learning_rate=0.03,
        num_leaves=63,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_STATE + seed_offset,
        objective="regression",
        verbosity=-1,
    )


def make_preprocessor(numeric_features, categorical_features):
    return ColumnTransformer(
        transformers=[
            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
            ("num", "passthrough", numeric_features),
        ],
        remainder="drop",
    )


def regression_metrics(y_true, y_pred):
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(root_mean_squared_error(y_true, y_pred)),
        "R2": float(r2_score(y_true, y_pred)),
    }


def train_models_for_horizon(horizon: int):
    targets = target_cols_by_horizon[horizon]
    target_rotation = targets["target_rotation"]
    target_speed = targets["target_speed"]

    rotation_model = Pipeline(
        steps=[
            ("preprocess", make_preprocessor(base_numeric_features, categorical_features)),
            ("model", make_regressor(seed_offset=horizon)),
        ]
    )

    rotation_model.fit(
        train_df[base_numeric_features + categorical_features],
        train_df[target_rotation],
    )

    pred_target_rotation = rotation_model.predict(
        test_df[base_numeric_features + categorical_features]
    )

    train_speed_df = train_df.copy()
    train_speed_df["candidate_target_rotation"] = train_speed_df[target_rotation]

    test_speed_oracle_df = test_df.copy()
    test_speed_oracle_df["candidate_target_rotation"] = test_speed_oracle_df[target_rotation]

    test_speed_chained_df = test_df.copy()
    test_speed_chained_df["candidate_target_rotation"] = pred_target_rotation

    speed_model = Pipeline(
        steps=[
            ("preprocess", make_preprocessor(speed_numeric_features, categorical_features)),
            ("model", make_regressor(seed_offset=100 + horizon)),
        ]
    )

    speed_model.fit(
        train_speed_df[speed_numeric_features + categorical_features],
        train_speed_df[target_speed],
    )

    pred_speed_oracle = speed_model.predict(
        test_speed_oracle_df[speed_numeric_features + categorical_features]
    )

    pred_speed_chained = speed_model.predict(
        test_speed_chained_df[speed_numeric_features + categorical_features]
    )

    return {
        "horizon": horizon,
        "rotation_model": rotation_model,
        "speed_model": speed_model,
        "pred_target_rotation": pred_target_rotation,
        "pred_speed_oracle": pred_speed_oracle,
        "pred_speed_chained": pred_speed_chained,
        "rotation_metrics": regression_metrics(test_df[target_rotation], pred_target_rotation),
        "speed_metrics_oracle": regression_metrics(test_df[target_speed], pred_speed_oracle),
        "speed_metrics_chained": regression_metrics(test_df[target_speed], pred_speed_chained),
    }


In [ ]:
models_by_horizon = {}
metrics_rows = []

for h in TARGET_HORIZONS:
    result = train_models_for_horizon(h)
    models_by_horizon[h] = result

    metrics_rows.extend(
        [
            {"horizon": h, "model": f"rotation_model_near{h}", **result["rotation_metrics"]},
            {"horizon": h, "model": f"speed_model_oracle_rotation_near{h}", **result["speed_metrics_oracle"]},
            {"horizon": h, "model": f"speed_model_chained_near{h}", **result["speed_metrics_chained"]},
        ]
    )

metrics_summary = pd.DataFrame(metrics_rows)
display(metrics_summary)



In [ ]:
baseline_rows = []

for h in TARGET_HORIZONS:
    target_speed = target_cols_by_horizon[h]["target_speed"]

    baseline_rows.extend(
        [
            {"horizon": h, "model": "current_speed", **regression_metrics(test_df[target_speed], test_df["speed"])},
            {
                "horizon": h,
                "model": "speed_roll_mean_12",
                **regression_metrics(
                    test_df[target_speed],
                    test_df["speed_roll_mean_12"].fillna(test_df["speed"]),
                ),
            },
            {
                "horizon": h,
                "model": f"rotation_to_speed_chained_near{h}",
                **models_by_horizon[h]["speed_metrics_chained"],
            },
            {
                "horizon": h,
                "model": f"speed_oracle_rotation_near{h}",
                **models_by_horizon[h]["speed_metrics_oracle"],
            },
        ]
    )

baseline_compare = pd.DataFrame(baseline_rows)
display(baseline_compare)

surface_ranges = {}

for et, part in train_df.groupby("rock_energy_type_final"):
    if len(part) < 100:
        continue

    surface_ranges[et] = {
        "pressure_axis_q05": float(part["pressure_axis"].quantile(0.05)),
        "pressure_axis_q95": float(part["pressure_axis"].quantile(0.95)),
        "pressure_rotation_q05": float(part["pressure_rotation"].quantile(0.05)),
        "pressure_rotation_q95": float(part["pressure_rotation"].quantile(0.95)),
        "rows": int(len(part)),
    }



In [ ]:
def drop_duplicate_columns(frame: pd.DataFrame) -> pd.DataFrame:
    return frame.loc[:, ~frame.columns.duplicated()].copy()


def recompute_candidate_features(grid: pd.DataFrame) -> pd.DataFrame:
    grid = grid.copy()
    grid["total_pressure"] = grid["pressure_axis"] + grid["pressure_rotation"]
    grid["pressure_balance"] = grid["pressure_axis"] / (grid["total_pressure"] + EPS)
    grid["rotation_efficiency"] = grid["rotation"] / (grid["pressure_rotation"] + EPS)
    grid["axis_x_rotation"] = grid["pressure_axis"] * grid["rotation"]
    grid["energy_input_proxy"] = grid["pressure_axis"] + grid["pressure_rotation"] * grid["rotation"]
    return grid


def build_candidate_grid_batch(rows: pd.DataFrame, grid_size: int = GRID_SIZE) -> pd.DataFrame:
    rows = rows.reset_index(drop=True).copy()
    n = len(rows)

    fractions = np.linspace(0.0, 1.0, grid_size)
    fa, fr = np.meshgrid(fractions, fractions)
    fa = fa.ravel()
    fr = fr.ravel()
    candidates_per_row = len(fa)

    repeated = rows.loc[rows.index.repeat(candidates_per_row)].reset_index(drop=True)

    cur_axis = rows["pressure_axis"].to_numpy(dtype=float)
    cur_rot = rows["pressure_rotation"].to_numpy(dtype=float)

    default_axis_low = float(train_df["pressure_axis"].quantile(0.05))
    default_axis_high = float(train_df["pressure_axis"].quantile(0.95))
    default_rot_low = float(train_df["pressure_rotation"].quantile(0.05))
    default_rot_high = float(train_df["pressure_rotation"].quantile(0.95))

    axis_low_global = np.array([
        surface_ranges.get(et, {}).get("pressure_axis_q05", default_axis_low)
        for et in rows["rock_energy_type_final"]
    ])
    axis_high_global = np.array([
        surface_ranges.get(et, {}).get("pressure_axis_q95", default_axis_high)
        for et in rows["rock_energy_type_final"]
    ])
    rot_low_global = np.array([
        surface_ranges.get(et, {}).get("pressure_rotation_q05", default_rot_low)
        for et in rows["rock_energy_type_final"]
    ])
    rot_high_global = np.array([
        surface_ranges.get(et, {}).get("pressure_rotation_q95", default_rot_high)
        for et in rows["rock_energy_type_final"]
    ])

    local_axis_low = cur_axis * (1.0 - MAX_DELTA_FRAC)
    local_axis_high = cur_axis * (1.0 + MAX_DELTA_FRAC)
    local_rot_low = cur_rot * (1.0 - MAX_DELTA_FRAC)
    local_rot_high = cur_rot * (1.0 + MAX_DELTA_FRAC)

    axis_min = np.maximum(axis_low_global, local_axis_low)
    axis_max = np.minimum(axis_high_global, local_axis_high)
    rot_min = np.maximum(rot_low_global, local_rot_low)
    rot_max = np.minimum(rot_high_global, local_rot_high)

    bad_axis = axis_min >= axis_max
    axis_min[bad_axis] = local_axis_low[bad_axis]
    axis_max[bad_axis] = local_axis_high[bad_axis]

    bad_rot = rot_min >= rot_max
    rot_min[bad_rot] = local_rot_low[bad_rot]
    rot_max[bad_rot] = local_rot_high[bad_rot]

    axis_values = np.repeat(axis_min, candidates_per_row) + np.tile(fa, n) * np.repeat(axis_max - axis_min, candidates_per_row)
    rot_values = np.repeat(rot_min, candidates_per_row) + np.tile(fr, n) * np.repeat(rot_max - rot_min, candidates_per_row)

    candidates = repeated[base_numeric_features + categorical_features + ["row_id"]].copy()
    candidates["pressure_axis"] = axis_values
    candidates["pressure_rotation"] = rot_values
    candidates = recompute_candidate_features(candidates)

    candidates["delta_pressure_axis_frac"] = axis_values / (np.repeat(cur_axis, candidates_per_row) + EPS) - 1.0
    candidates["delta_pressure_rotation_frac"] = rot_values / (np.repeat(cur_rot, candidates_per_row) + EPS) - 1.0

    return drop_duplicate_columns(candidates)


def score_candidates(grid: pd.DataFrame) -> pd.DataFrame:
    grid = grid.copy()

    speed_scale = np.maximum(
        np.abs(grid["current_predicted_target_speed"].to_numpy(dtype=float)),
        EPS,
    )

    axis_delta_norm = np.abs(grid["delta_pressure_axis_frac"].to_numpy(dtype=float)) / (MAX_DELTA_FRAC + EPS)
    rot_delta_norm = np.abs(grid["delta_pressure_rotation_frac"].to_numpy(dtype=float)) / (MAX_DELTA_FRAC + EPS)

    change_penalty = (
        CHANGE_PENALTY_WEIGHT
        * speed_scale
        * (axis_delta_norm ** 2 + rot_delta_norm ** 2)
    )

    axis_boundary_excess = np.maximum(
        0.0,
        (axis_delta_norm - BOUNDARY_START) / (1.0 - BOUNDARY_START + EPS),
    )
    rot_boundary_excess = np.maximum(
        0.0,
        (rot_delta_norm - BOUNDARY_START) / (1.0 - BOUNDARY_START + EPS),
    )

    boundary_penalty = (
        BOUNDARY_PENALTY_WEIGHT
        * speed_scale
        * (axis_boundary_excess ** 2 + rot_boundary_excess ** 2)
    )

    grid["change_penalty"] = change_penalty
    grid["boundary_penalty"] = boundary_penalty
    grid["optimizer_score"] = (
        grid["predicted_target_speed"].to_numpy(dtype=float)
        - change_penalty
        - boundary_penalty
    )

    return grid



## Non-overlap well-level time report

Оценка времени по скважинам теперь считается по непересекающимся near_5-блокам.

Для строки `t` прогноз `near_5` относится к блоку `t+1 ... t+5`. Следующий используемый блок начинается только после него, поэтому будущие интервалы не переиспользуются в соседних расчётах.


In [ ]:
base_feature_cols = base_numeric_features + categorical_features
speed_feature_cols = speed_numeric_features + categorical_features

print("Base feature columns:", len(base_feature_cols))
print("Speed feature columns:", len(speed_feature_cols))


In [ ]:
REPORT_DIR = PROJECT_ROOT / "notebooks" / "model_reports"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

WELL_REPORT_HORIZON = DEFAULT_HORIZON
WELL_REPORT_CHUNK_ROWS = 5000

# Non-overlap near_5 report:
# each evaluation row t represents the next block t+1 ... t+5.
# The next used row starts after that block, so future windows do not overlap.

h = WELL_REPORT_HORIZON
target_speed_col = target_cols_by_horizon[h]["target_speed"]

well_source_df = test_df.copy().sort_values(["well_id", "processing_time"]).reset_index(drop=True)
well_source_df = well_source_df[
    (well_source_df["d_time"] > 0)
    & (well_source_df["d_depth"] > 0)
    & (well_source_df["speed"] > 0)
].copy()

# Build non-overlapping block ids inside every test well.
well_source_df["_pos_in_well"] = well_source_df.groupby("well_id").cumcount()
well_source_df["_block_id"] = (well_source_df["_pos_in_well"] // h).astype(int)
well_source_df["_pos_in_block"] = (well_source_df["_pos_in_well"] % h).astype(int)

# For a prediction made at row t, the evaluated physical interval is t+1 ... t+h.
# Therefore we use only rows where a full future block exists.
# Anchor rows are the row immediately before each future block:
# block 1 anchor -> last row of block 0, block 2 anchor -> last row of block 1, etc.
anchor_df = well_source_df[well_source_df["_pos_in_block"] == h - 1].copy()
anchor_df["_future_block_id"] = anchor_df["_block_id"] + 1

future_block_df = (
    well_source_df
    .groupby(["well_id", "_block_id"])
    .agg(
        block_start=("processing_time", "min"),
        block_end=("processing_time", "max"),
        block_rows=("speed", "size"),
        block_depth_m=("d_depth", "sum"),
        operator_block_time_sec=("d_time", "sum"),
        operator_block_speed=("speed", lambda s: float(np.average(s, weights=well_source_df.loc[s.index, "d_time"]))),
        future_pressure_axis_mean=("pressure_axis", "mean"),
        future_pressure_rotation_mean=("pressure_rotation", "mean"),
    )
    .reset_index()
    .rename(columns={"_block_id": "_future_block_id"})
)

# Keep only full future 5-step blocks.
future_block_df = future_block_df[future_block_df["block_rows"] == h].copy()

anchor_df = anchor_df.merge(
    future_block_df,
    on=["well_id", "_future_block_id"],
    how="inner",
)

anchor_df = anchor_df[
    (anchor_df["block_depth_m"] > 0)
    & (anchor_df["operator_block_time_sec"] > 0)
].copy()

anchor_df = anchor_df.reset_index(drop=True)
anchor_df["row_id"] = np.arange(len(anchor_df))

print("Non-overlap near_5 anchors:", len(anchor_df))
print("Test wells with full non-overlap blocks:", anchor_df["well_id"].nunique())
print("Physical rows covered by blocks:", int(anchor_df["block_rows"].sum()))

rotation_model = models_by_horizon[WELL_REPORT_HORIZON]["rotation_model"]
speed_model = models_by_horizon[WELL_REPORT_HORIZON]["speed_model"]

best_chunks = []
current_pred_chunks = []
candidates_per_row = GRID_SIZE * GRID_SIZE

for start in range(0, len(anchor_df), WELL_REPORT_CHUNK_ROWS):
    chunk = anchor_df.iloc[start:start + WELL_REPORT_CHUNK_ROWS].copy()

    current_chunk = chunk.copy()
    current_chunk["candidate_target_rotation"] = rotation_model.predict(current_chunk[base_feature_cols])
    current_pred = speed_model.predict(current_chunk[speed_feature_cols])

    current_pred_chunks.append(
        pd.DataFrame({
            "row_id": chunk["row_id"].to_numpy(),
            "current_predicted_target_speed": current_pred,
        })
    )

    candidates = build_candidate_grid_batch(chunk, grid_size=GRID_SIZE)
    candidates["candidate_target_rotation"] = rotation_model.predict(candidates[base_feature_cols])
    candidates["predicted_target_speed"] = speed_model.predict(candidates[speed_feature_cols])
    candidates["current_predicted_target_speed"] = np.repeat(current_pred, candidates_per_row)

    candidates = score_candidates(candidates)

    scores = candidates["optimizer_score"].to_numpy(dtype=float).reshape(len(chunk), candidates_per_row)
    best_offsets = scores.argmax(axis=1)
    best_idx = np.arange(len(chunk)) * candidates_per_row + best_offsets

    best_chunks.append(
        candidates.iloc[best_idx][
            [
                "row_id",
                "pressure_axis",
                "pressure_rotation",
                "candidate_target_rotation",
                "predicted_target_speed",
                "optimizer_score",
                "delta_pressure_axis_frac",
                "delta_pressure_rotation_frac",
            ]
        ].copy()
    )

    print(f"processed anchors: {min(start + WELL_REPORT_CHUNK_ROWS, len(anchor_df))} / {len(anchor_df)}")

best_recs = pd.concat(best_chunks, ignore_index=True)
current_pred_df = pd.concat(current_pred_chunks, ignore_index=True)

block_level_time = (
    anchor_df
    .merge(best_recs, on="row_id", how="left", suffixes=("", "_recommended"))
    .merge(current_pred_df, on="row_id", how="left")
)

block_level_time = block_level_time.rename(
    columns={
        "pressure_axis_recommended": "recommended_pressure_axis",
        "pressure_rotation_recommended": "recommended_pressure_rotation",
        "candidate_target_rotation": "recommended_candidate_target_rotation",
        "predicted_target_speed": "recommended_predicted_speed",
    }
)

target_speed_train = train_df[target_speed_col]
speed_clip_low = float(target_speed_train.quantile(0.01))
speed_clip_high = float(target_speed_train.quantile(0.99))

block_level_time["recommended_predicted_speed_clipped"] = block_level_time["recommended_predicted_speed"].clip(
    lower=speed_clip_low,
    upper=speed_clip_high,
)

# Non-overlap block time calculation:
# model uses one near_5 predicted mean speed for exactly one non-overlapping future block.
block_level_time["model_block_time_sec"] = (
    block_level_time["block_depth_m"]
    / (block_level_time["recommended_predicted_speed_clipped"] + EPS)
)
block_level_time["operator_block_time_min"] = block_level_time["operator_block_time_sec"] / 60.0
block_level_time["model_block_time_min"] = block_level_time["model_block_time_sec"] / 60.0

block_level_time["block_time_saved_min"] = (
    block_level_time["operator_block_time_min"]
    - block_level_time["model_block_time_min"]
)
block_level_time["block_time_saved_pct"] = 100.0 * (
    block_level_time["block_time_saved_min"]
    / (block_level_time["operator_block_time_min"] + EPS)
)

block_level_time["model_based_predicted_uplift_pct"] = 100.0 * (
    block_level_time["recommended_predicted_speed"]
    / (block_level_time["current_predicted_target_speed"] + EPS)
    - 1.0
)
block_level_time["predicted_regret_vs_operator_pct"] = 100.0 * (
    block_level_time["recommended_predicted_speed"]
    / (block_level_time["operator_block_speed"] + EPS)
    - 1.0
)
block_level_time["recommended_win_vs_operator"] = (
    block_level_time["recommended_predicted_speed"] > block_level_time["operator_block_speed"]
)

# Energy-class shares by non-overlapping future block depth.
# We assign the anchor's current energy type to the evaluated block. For well-level composition,
# this is used only as context, not as a causal metric.
energy_labels = [
    "soft_low_energy",
    "medium_low_energy",
    "medium_high_energy",
    "hard_high_energy",
]

energy_depth = (
    block_level_time
    .groupby(["well_id", "rock_energy_type_final"])["block_depth_m"]
    .sum()
    .reset_index()
)

energy_pivot = energy_depth.pivot_table(
    index="well_id",
    columns="rock_energy_type_final",
    values="block_depth_m",
    aggfunc="sum",
    fill_value=0.0,
)

for label in energy_labels:
    if label not in energy_pivot.columns:
        energy_pivot[label] = 0.0

energy_pivot["energy_total_depth_m"] = energy_pivot[energy_labels].sum(axis=1)

for label in energy_labels:
    energy_pivot[f"{label}_pct"] = 100.0 * energy_pivot[label] / (energy_pivot["energy_total_depth_m"] + EPS)

energy_share_cols = [f"{label}_pct" for label in energy_labels]
energy_pivot = energy_pivot.reset_index()[["well_id"] + energy_share_cols]

well_time_report = (
    block_level_time
    .groupby("well_id")
    .agg(
        non_overlap_blocks=("row_id", "size"),
        telemetry_rows_covered=("block_rows", "sum"),
        drilling_start=("block_start", "min"),
        drilling_end=("block_end", "max"),
        total_depth_modeled_m=("block_depth_m", "sum"),
        operator_time_sec=("operator_block_time_sec", "sum"),
        model_time_sec=("model_block_time_sec", "sum"),
        operator_block_speed_mean=("operator_block_speed", "mean"),
        recommended_predicted_speed_mean=("recommended_predicted_speed_clipped", "mean"),
        current_predicted_speed_mean=("current_predicted_target_speed", "mean"),
        current_pressure_axis_mean=("pressure_axis", "mean"),
        current_pressure_rotation_mean=("pressure_rotation", "mean"),
        recommended_pressure_axis_mean=("recommended_pressure_axis", "mean"),
        recommended_pressure_rotation_mean=("recommended_pressure_rotation", "mean"),
        model_based_predicted_uplift_pct_median=("model_based_predicted_uplift_pct", "median"),
        predicted_regret_vs_operator_pct_median=("predicted_regret_vs_operator_pct", "median"),
        recommended_win_vs_operator_rate=("recommended_win_vs_operator", "mean"),
    )
    .reset_index()
    .merge(energy_pivot, on="well_id", how="left")
)

well_time_report[energy_share_cols] = well_time_report[energy_share_cols].fillna(0.0)

well_time_report["operator_time_min"] = well_time_report["operator_time_sec"] / 60.0
well_time_report["model_time_min"] = well_time_report["model_time_sec"] / 60.0
well_time_report["time_saved_min"] = well_time_report["operator_time_min"] - well_time_report["model_time_min"]
well_time_report["time_saved_pct"] = 100.0 * well_time_report["time_saved_min"] / (well_time_report["operator_time_min"] + EPS)

well_time_report = well_time_report[
    [
        "well_id",
        "non_overlap_blocks",
        "telemetry_rows_covered",
        "drilling_start",
        "drilling_end",
        "total_depth_modeled_m",
        *energy_share_cols,
        "operator_time_min",
        "model_time_min",
        "time_saved_min",
        "time_saved_pct",
        "operator_block_speed_mean",
        "current_predicted_speed_mean",
        "recommended_predicted_speed_mean",
        "model_based_predicted_uplift_pct_median",
        "predicted_regret_vs_operator_pct_median",
        "recommended_win_vs_operator_rate",
        "current_pressure_axis_mean",
        "recommended_pressure_axis_mean",
        "current_pressure_rotation_mean",
        "recommended_pressure_rotation_mean",
    ]
].sort_values("well_id")

well_time_summary = pd.DataFrame([
    {
        "horizon": WELL_REPORT_HORIZON,
        "method": "non_overlap_near5",
        "wells": int(len(well_time_report)),
        "non_overlap_blocks": int(len(block_level_time)),
        "telemetry_rows_covered": int(block_level_time["block_rows"].sum()),
        "operator_time_total_min": float(well_time_report["operator_time_min"].sum()),
        "model_time_total_min": float(well_time_report["model_time_min"].sum()),
        "time_saved_total_min": float(well_time_report["time_saved_min"].sum()),
        "time_saved_total_pct": float(
            100.0
            * well_time_report["time_saved_min"].sum()
            / (well_time_report["operator_time_min"].sum() + EPS)
        ),
        "median_well_time_saved_pct": float(well_time_report["time_saved_pct"].median()),
        "mean_well_time_saved_pct": float(well_time_report["time_saved_pct"].mean()),
        "p05_well_time_saved_pct": float(well_time_report["time_saved_pct"].quantile(0.05)),
        "p95_well_time_saved_pct": float(well_time_report["time_saved_pct"].quantile(0.95)),
        "median_block_model_based_uplift_pct": float(block_level_time["model_based_predicted_uplift_pct"].median()),
        "median_block_predicted_regret_vs_operator_pct": float(block_level_time["predicted_regret_vs_operator_pct"].median()),
        "block_recommended_win_vs_operator_rate": float(block_level_time["recommended_win_vs_operator"].mean()),
    }
])


display(well_time_summary)
display(well_time_report.head())

block_level_time.to_csv(REPORT_DIR / "near5_block_offline_evaluation.csv", index=False)
well_time_report.to_csv(REPORT_DIR / "near5_well_offline_evaluation.csv", index=False)


In [ ]:
ARTIFACT_DIR = PROJECT_ROOT / "notebooks" / "model_artifacts"
REPORT_DIR = PROJECT_ROOT / "notebooks" / "model_reports"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

for h, result in models_by_horizon.items():
    joblib.dump(result["rotation_model"], ARTIFACT_DIR / f"rotation_model_near{h}.joblib")
    joblib.dump(result["speed_model"], ARTIFACT_DIR / f"speed_model_near{h}.joblib")


with open(ARTIFACT_DIR / "candidate_pressure_ranges_by_energy_type.json", "w", encoding="utf-8") as f:
    json.dump(surface_ranges, f, ensure_ascii=False, indent=2)

feature_config = {
    "data_path": str(DATA_PATH),
    "target_horizons": TARGET_HORIZONS,
    "default_horizon": DEFAULT_HORIZON,
    "base_numeric_features": base_numeric_features,
    "categorical_features": categorical_features,
    "rotation_features": base_numeric_features + categorical_features,
    "speed_features": speed_numeric_features + categorical_features,
    "speed_extra_features": speed_extra_features,
    "hardness_feature_columns": HARDNESS_FEATURE_COLUMNS,
    "energy_type_column": "rock_energy_type_final",
    "depth_column": "depth",
    "d_depth_column": "d_depth",
    "d_time_column": "d_time",
}

with open(ARTIFACT_DIR / "feature_config.json", "w", encoding="utf-8") as f:
    json.dump(feature_config, f, ensure_ascii=False, indent=2)

optimizer_config = {
    "optimizer_mode": FINAL_OPTIMIZER_MODE,
    "target_horizons": TARGET_HORIZONS,
    "default_horizon": DEFAULT_HORIZON,
    "grid_size_default": int(GRID_SIZE),
    "max_delta_frac_default": float(MAX_DELTA_FRAC),
    "change_penalty_weight": float(CHANGE_PENALTY_WEIGHT),
    "boundary_penalty_weight": float(BOUNDARY_PENALTY_WEIGHT),
    "boundary_start": float(BOUNDARY_START),
    "score_formula": "pred_target_speed - change_penalty - boundary_penalty",
    "well_level_method": "non_overlap_near5",
    "well_level_formula": "model_block_time = sum(d_depth over t+1...t+5) / recommended_predicted_speed_near5_at_anchor",
    "operator_comparison_interpretation": "offline counterfactual diagnostics; not a measured causal effect",
}

with open(ARTIFACT_DIR / "optimizer_config.json", "w", encoding="utf-8") as f:
    json.dump(optimizer_config, f, ensure_ascii=False, indent=2)

metrics_summary.to_csv(REPORT_DIR / "near5_model_metrics.csv", index=False)
baseline_compare.to_csv(REPORT_DIR / "near5_baseline_metrics.csv", index=False)
well_time_summary.to_csv(REPORT_DIR / "near5_offline_summary.csv", index=False)

print("Saved artifacts:", ARTIFACT_DIR)
print("Saved reports:", REPORT_DIR)
display(metrics_summary)
display(well_time_summary)
